# Install Libraries

In [ ]:
%pip install gymnasium numpy matplotlib

# FrozenLake — Tabular Q-Learning

## The Game
FrozenLake is a 4×4 grid world. The agent starts at the top-left (S) and must reach the goal (G) without falling into holes (H). The ice is slippery, so actions don't always move in the intended direction.

```
SFFF
FHFH
FFFH
HFFG
```

## RL Formulation
- **State**: integer 0–15 (row × 4 + col)
- **Action**: 0=Left, 1=Down, 2=Right, 3=Up
- **Reward**: +1 on reaching G, 0 otherwise

## Algorithm: Tabular Q-Learning
We maintain a table `Q[state, action]` updated by the Bellman equation:

$$Q(s,a) \leftarrow Q(s,a) + \alpha \left[ r + \gamma \max_{a'} Q(s',a') - Q(s,a) \right]$$

Actions are selected **epsilon-greedy**: random with probability ε, greedy otherwise. ε decays over training so the agent explores early and exploits later.

# Create & Test Environment

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

def test_env(env, num_steps=20):
    obs, info = env.reset()
    print(f"Initial state: {obs}")
    total_reward = 0
    for step in range(num_steps):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        action_name = ['Left','Down','Right','Up'][action]
        print(f"Step {step+1}: action={action_name}, state={obs}, reward={reward}, done={terminated}")
        if terminated or truncated:
            print(f"Episode ended. Total reward: {total_reward}")
            break
    env.close()

env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True)
print('Observation space:', env.observation_space)
print('Action space:', env.action_space)
test_env(env)

# Train Agent

In [ ]:
import os

# Hyperparameters
N_EPISODES    = 15_000
ALPHA         = 0.8
GAMMA         = 0.95
EPSILON_START = 1.0
EPSILON_END   = 0.05
EPSILON_DECAY = 0.9997

# Checkpoints at 0%, 25%, 50%, 75%, 100%
CHECKPOINT_EPISODES = [0, N_EPISODES // 4, N_EPISODES // 2, 3 * N_EPISODES // 4, N_EPISODES - 1]
CHECKPOINT_LABELS   = ['Untrained (ep 0)', '25%', '50%', '75%', '100% (fully trained)']
os.makedirs('models/frozenlake', exist_ok=True)

env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True)
n_states  = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))
epsilon = EPSILON_START
episode_rewards = []

for episode in range(N_EPISODES):
    # Save checkpoint before the episode runs
    if episode in CHECKPOINT_EPISODES:
        idx = CHECKPOINT_EPISODES.index(episode)
        path = f'models/frozenlake/qtable_ep{episode}.npy'
        np.save(path, Q)
        print(f'Saved checkpoint [{CHECKPOINT_LABELS[idx]}] -> {path}')

    state, _ = env.reset()
    total_reward = 0

    while True:
        if np.random.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state])

        next_state, reward, terminated, truncated, _ = env.step(action)
        Q[state, action] += ALPHA * (reward + GAMMA * np.max(Q[next_state]) - Q[state, action])
        state = next_state
        total_reward += reward

        if terminated or truncated:
            break

    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    episode_rewards.append(total_reward)

    if (episode + 1) % 1000 == 0:
        avg = np.mean(episode_rewards[-1000:])
        print(f'Episode {episode+1:6d} | ε={epsilon:.3f} | win rate (last 1k): {avg:.3f}')

# Save final checkpoint
np.save(f'models/frozenlake/qtable_ep{N_EPISODES-1}.npy', Q)
env.close()

# Evaluate & Visualize

In [ ]:
window = 500
moving_avg = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')

plt.figure(figsize=(10, 4))
plt.plot(moving_avg)
for ep, label in zip(CHECKPOINT_EPISODES[1:], CHECKPOINT_LABELS[1:]):
    x = min(ep, len(moving_avg) - 1)
    plt.axvline(x, color='red', linestyle='--', alpha=0.5, label=label)
plt.xlabel('Episode')
plt.ylabel(f'Win rate (moving avg, window={window})')
plt.title('FrozenLake — Q-Learning Training Progress')
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
def eval_qtable(Q, n_episodes=1000):
    env = gym.make('FrozenLake-v1', map_name='4x4', is_slippery=True)
    wins = 0
    for _ in range(n_episodes):
        state, _ = env.reset()
        while True:
            action = np.argmax(Q[state])
            state, reward, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                wins += reward
                break
    env.close()
    return wins / n_episodes

print('Greedy win rate over 1000 episodes:')
print(f'  {CHECKPOINT_LABELS[-1]}: {eval_qtable(Q):.3f}')

# Compare: Untrained vs Intermediate vs Fully Trained

We load each saved Q-table checkpoint and evaluate its greedy win rate over 1000 episodes.

In [ ]:
checkpoint_paths = [f'models/frozenlake/qtable_ep{ep}.npy' for ep in CHECKPOINT_EPISODES]
win_rates = []

for path, label in zip(checkpoint_paths, CHECKPOINT_LABELS):
    q = np.load(path)
    wr = eval_qtable(q, n_episodes=1000)
    win_rates.append(wr)
    print(f'  {label:30s}: win rate = {wr:.3f}')

# Bar chart comparison
colors = ['#d62728', '#ff7f0e', '#bcbd22', '#2ca02c', '#1f77b4']
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(CHECKPOINT_LABELS, win_rates, color=colors, edgecolor='black', linewidth=0.8)
for bar, wr in zip(bars, win_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{wr:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_ylabel('Win Rate (greedy, 1000 episodes)')
ax.set_title('FrozenLake — Win Rate at Each Training Checkpoint')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize greedy policy arrows for untrained vs fully trained
action_symbols = ['←', '↓', '→', '↑']
MAP = ['SFFF', 'FHFH', 'FFFH', 'HFFG']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
qs_to_show = [np.load(checkpoint_paths[0]), np.load(checkpoint_paths[-1])]
titles = [f'Policy — {CHECKPOINT_LABELS[0]}', f'Policy — {CHECKPOINT_LABELS[-1]}']

for ax, q, title in zip(axes, qs_to_show, titles):
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(-0.5, 3.5)
    ax.set_xticks(range(4))
    ax.set_yticks(range(4))
    ax.grid(True)
    ax.set_title(title)
    for state in range(16):
        r, c = divmod(state, 4)
        cell = MAP[r][c]
        color = 'lightcoral' if cell == 'H' else ('lightgreen' if cell == 'G' else 'white')
        ax.add_patch(plt.Rectangle((c-0.5, 3-r-0.5), 1, 1, color=color))
        label = cell if cell in ('H', 'G', 'S') else action_symbols[np.argmax(q[state])]
        ax.text(c, 3-r, label, ha='center', va='center', fontsize=18)

plt.suptitle('FrozenLake — Policy Comparison', fontsize=13)
plt.tight_layout()
plt.show()